In [55]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import geopandas as gpd
import plotly.graph_objects as go
import matplotlib as mpl

mpl.use("pgf")
plt.rcParams['pgf.texsystem'] = 'pdflatex'
plt.rcParams['text.usetex'] = True
plt.rcParams['pgf.rcfonts'] = False
plt.rcParams['figure.edgecolor'] = 'k'
plt.rcParams['figure.facecolor'] = 'w'
plt.rcParams['savefig.dpi'] = 600
plt.rcParams['savefig.bbox'] = 'tight'
plt.rcParams['font.family'] = "serif"
plt.rcParams['axes.labelsize'] = 18
plt.rcParams['xtick.labelsize'] = 16
plt.rcParams['ytick.labelsize'] = 16

In [79]:
df = pd.read_csv('../../supplemental_materials/data_extraction.csv',
                 usecols=['AI Type - Intervention', 'Comparison', 'Country', 'Data Type', 'Decision Making - (Intervention)', 'Deployed?', 'Feature Study?', 'Industry', 'Output Feature', 'Published Year', 'QA Score', 'Regulatory Oversight Body (if any)', 'Safety Outcome', 'Study','Subcategory', 'Was explainability/interpretability considered?', 'XAI Type', 'Was regulation considered?', 'Decision-Making Time', 'Expertise Level of User']
                )

In [80]:
df = df[["Decision-Making Time","Expertise Level of User", "Was explainability/interpretability considered?", "Decision Making - (Intervention)"]]

In [86]:
col_map = {
    'Decision-Making Time': 'Time',
    'Expertise Level of User': 'Expertise',
    'Was explainability/interpretability considered?': 'Explainability',
    'Decision Making - (Intervention)': 'Decision-Type'
}
df = df.rename(columns=col_map)
df = df.fillna('Non Specific')
df = df.replace({"Augmented Human Decision-Making":"Augmented Human"})
# df = df.replace({"Static ML-Informed Benchmark": "Other", "Non Applicable": "Other", "Not System Integrated":"Other"})
df.head()

,Time,Expertise,Explainability,Decision-Type
0,Minutes-Hours,Expert,Yes,Augmented Human
1,Days or More,Expert,Yes,Augmented Human
2,Minutes-Hours,Expert,Yes,Static ML-Informed Benchmark
3,Days or More,Expert,Yes,Human in the Loop
4,Days or More,Expert,Yes,Static ML-Informed Benchmark


In [87]:
nodes = []
for col in df.columns:
    nodes += list(df[col].unique())
print(nodes)

['Minutes-Hours', 'Days or More', 'Seconds or Less', 'Non Specific', 'Expert', 'General Public', 'Trained User', 'Non Specific', 'Yes', 'No', 'Augmented Human', 'Static ML-Informed Benchmark', 'Human in the Loop', 'Autonomous', 'Non Applicable', 'Not System Integrated']


In [88]:
stages = ["Expertise", "Time", "Decision-Type", "Explainability"]
print(stages)

['Expertise', 'Time', 'Decision-Type', 'Explainability']


In [89]:
node_labels = []
node_lookup = {}  # (stage_idx, category_value) -> node index
 
for stage_idx, stage in enumerate(stages):
    for value in df[stage].unique():
        node_lookup[(stage_idx, value)] = len(node_labels)
        node_labels.append(str(value))
 

source, target, value = [], [], []
 
for stage_idx in range(len(stages) - 1):
    left_col = stages[stage_idx]
    right_col = stages[stage_idx + 1]
 
    flow_counts = df.groupby([left_col, right_col]).size().reset_index(name="count")
 
    for _, row in flow_counts.iterrows():
        left_val, right_val, count = row[left_col], row[right_col], row["count"]
        source.append(node_lookup[(stage_idx, left_val)])
        target.append(node_lookup[(stage_idx + 1, right_val)])
        value.append(count)

In [90]:
# set up labeling stuff
stage_x = [0, 0.33, 0.67, 1.0]
node_x = []
for stage_idx, stage in enumerate(stages):
    for _ in df[stage].unique():
        node_x.append(stage_x[stage_idx])

In [97]:
fig = go.Figure(
    data=[
        go.Sankey(
            node=dict(
                pad=15,
                thickness=18,
                line=dict(color="black", width=0.5),
                label=node_labels,
            ),
            link=dict(source=source, target=target, value=value),
        )
    ]
)

stage_titles = ["Expertise", "Time", "Decision Type", "Explainability <br> Considered?"]
for x, title in zip(stage_x, stage_titles):
    fig.add_annotation(
        x=x, y=1.08, xref="paper", yref="paper",
        text=f"<b>{title}</b>", showarrow=False, font=dict(size=13, family="DejaVu Serif"),
    )

# fig.update_layout(
#     title_text="Article Flow: Expertise -> Time -> Decision-Making Type -> Explainability",
#     font_size=11,
# )

fig.update_layout(
    font=dict(family="DejaVu Serif", size=13),
)
 
fig.write_html("sankey.html") 
fig.write_image("sankey_withsubcats.pdf", scale=3)

In [92]:
cross_tab_df = pd.crosstab(df["Time"], df["Expertise"]).reindex(columns=["Expert", "Trained User", "General Public", "Non Specific"]).reindex(["Days or More", "Minutes-Hours", "Seconds or Less", "Non Specific"], axis=0)
cross_tab_df

Expertise,Expert,Trained User,General Public,Non Specific
Time,,,,
Days or More,16,1,0,0
Minutes-Hours,17,3,1,0
Seconds or Less,6,2,14,0
Non Specific,2,0,0,10


In [93]:
print(cross_tab_df.to_latex())

\begin{tabular}{lrrrr}
\toprule
Expertise & Expert & Trained User & General Public & Non Specific \\
Time &  &  &  &  \\
\midrule
Days or More & 16 & 1 & 0 & 0 \\
Minutes-Hours & 17 & 3 & 1 & 0 \\
Seconds or Less & 6 & 2 & 14 & 0 \\
Non Specific & 2 & 0 & 0 & 10 \\
\bottomrule
\end{tabular}



In [94]:
cross_tab_df.sum()

Expertise
Expert            41
Trained User       6
General Public    15
Non Specific      10
dtype: int64

In [95]:
41+6+15+10

72